# Restaurant AI - Model Training

This notebook covers:
- Creating custom dataset
- Data augmentation
- Training YOLOv8 model
- Model evaluation
- Export for deployment

In [ ]:
import sys
sys.path.append('..')

import os
import cv2
import numpy as np
from pathlib import Path
import yaml
import shutil

## 1. Setup

In [ ]:
from utils.train import RestaurantDataset, DataAugmentor, TrainingConfig
from ultralytics import YOLO

print("Imports successful")

## 2. Create Dataset

In [ ]:
OUTPUT_DIR = '../data/restaurant'
Path(OUTPUT_DIR).mkdir(exist_ok=True, parents=True)

dataset = RestaurantDataset(OUTPUT_DIR)
dataset.create_structure()
print(f"Dataset structure created at {OUTPUT_DIR}")

## 3. Generate Synthetic Training Data

In [ ]:
import random

def generate_synthetic_dataset(dataset, num_images=100):
    """Generate synthetic images with bounding boxes."""
    augmentor = DataAugmentor()
    
    for i in range(num_images):
        w, h = 640, 480
        
        img = np.random.randint(80, 200, (h, w, 3), dtype=np.uint8)
        
        num_persons = random.randint(1, 5)
        bboxes = []
        
        for _ in range(num_persons):
            x1 = random.randint(20, w - 150)
            y1 = random.randint(20, h - 200)
            x2 = x1 + random.randint(40, 120)
            y2 = y1 + random.randint(80, 180)
            
            color = (random.randint(30, 200), random.randint(30, 200), random.randint(30, 200))
            cv2.rectangle(img, (x1, y1), (x2, y2), color, -1)
            
            head_x = x1 + (x2 - x1) // 2
            head_y = y1 + 20
            cv2.circle(img, (head_x, head_y), 15, (220, 180, 160), -1)
            
            bboxes.append([x1, y1, x2, y2])
        
        split = 'train' if i < num_images * 0.8 else 'val'
        
        img_path = dataset.images_path / split / f'img_{i:04d}.jpg'
        cv2.imwrite(str(img_path), img)
        
        label_path = dataset.labels_path / split / f'img_{i:04d}.txt'
        with open(label_path, 'w') as f:
            for bbox in bboxes:
                cx = ((bbox[0] + bbox[2]) / 2) / w
                cy = ((bbox[1] + bbox[3]) / 2) / h
                bw = (bbox[2] - bbox[0]) / w
                bh = (bbox[3] - bbox[1]) / h
                f.write(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")
    
    print(f"Generated {num_images} images")

generate_synthetic_dataset(dataset, num_images=50)

## 4. Create Data Configuration

In [ ]:
config = TrainingConfig(model_size='n')
config.create_data_yaml('data.yaml')

print("data.yaml created:")
with open('data.yaml', 'r') as f:
    print(f.read())

## 5. Train Model

In [ ]:
def train_model(epochs=50, batch=8, imgsz=640):
    """Train the model."""
    model = YOLO('yolov8n.pt')
    
    results = model.train(
        data='data.yaml',
        epochs=epochs,
        batch=batch,
        imgsz=imgsz,
        name='restaurant_ai',
        project='../outputs/train',
        exist_ok=True,
        verbose=True,
        device='cpu',
        save=True
    )
    
    return results

# Uncomment to train:
# results = train_model(epochs=50)

## 6. Evaluate Model

In [ ]:
def evaluate_model(weights_path):
    """Evaluate trained model."""
    model = YOLO(weights_path)
    metrics = model.val()
    
    return {
        'map50': metrics.box.map50,
        'map': metrics.box.map,
        'precision': metrics.box.mp,
        'recall': metrics.box.mr
    }

# Uncomment to evaluate:
# metrics = evaluate_model('../outputs/train/restaurant_ai/weights/best.pt')

## 7. Export Model

In [ ]:
def export_model(weights_path, format='onnx'):
    """Export model to different formats."""
    model = YOLO(weights_path)
    exported = model.export(format=format)
    return exported

# Uncomment to export:
# export_model('../outputs/train/restaurant_ai/weights/best.pt', 'onnx')

## 8. Use Custom Model

In [ ]:
def use_custom_model(weights_path, image_path):
    """Use custom trained model for inference."""
    model = YOLO(weights_path)
    results = model(image_path)
    
    for r in results:
        print(f"Detected: {len(r.boxes)} objects")
        for box in r.boxes:
            print(f"  Class: {int(box.cls)}, Conf: {box.conf[0]:.2f}")
    
    return results

# Example:
# use_custom_model('../outputs/train/restaurant_ai/weights/best.pt', 'test.jpg')

## Summary

Steps to train your model:

1. **Collect Data**: Gather images from your restaurant CCTV
2. **Annotate**: Use tools like LabelImg or CVAT to annotate
3. **Augment**: Use DataAugmentor for more training data
4. **Train**: Run training with desired epochs
5. **Evaluate**: Check metrics (map50, precision, recall)
6. **Export**: Export to ONNX/TFLite for deployment
7. **Deploy**: Use in detection pipeline